In [ ]:
import os
from ultralytics import YOLO
import cv2

In [ ]:
def train_models():
    # Model variants to compare
    models_to_train = ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]

    for model_name in models_to_train:
        print(f"--- Starting Training for {model_name} ---")
        model = YOLO(model_name)
        model.train(
            data="data/rps/data.yaml", # Path to your Roboflow dataset
            epochs=10,
            imgsz=416,
            batch=16,
            device=0,  # Uses GPU
            name=model_name.replace(".pt", ""),
            workers=4
        )


In [ ]:
def evaluate_models():
    # Using the 'best' weights from your training runs
    model_paths = {
        "YOLOv8n": "runs/detect/yolov8n3/weights/best.pt",
        "YOLOv8s": "runs/detect/yolov8s/weights/best.pt",
        "YOLOv8m": "runs/detect/yolov8m2/weights/best.pt"
    }
    
    print(f"{'Model':<10} | {'mAP50':<8} | {'mAP50-95':<10} | {'Precision':<10} | {'Recall':<8}")
    print("-" * 65)

    for name, path in model_paths.items():
        try:
            model = YOLO(path)
            # Validate on the validation set defined in data.yaml
            results = model.val(data="data/rps/data.yaml", imgsz=416, plots=False)
            
            # Extract core metrics for the Comparison Table
            map50 = results.box.map50
            map95 = results.box.map
            prec = results.box.mp
            recall = results.box.mr
            
            print(f"{name:<10} | {map50:<8.3f} | {map95:<10.3f} | {prec:<10.3f} | {recall:<8.3f}")
        except Exception as e:
            print(f"Could not evaluate {name}: {e}")

evaluate_models()

In [ ]:
def run_live_demo():
    model = YOLO("runs/detect/yolov8n3/weights/best.pt")
    cap = cv2.VideoCapture(0)

    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        results = model(frame, conf=0.5, stream=True)
        for r in results:
            annotated_frame = r.plot()
            cv2.imshow("Rock Paper Scissors - Live Detection", annotated_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    cv2.destroyAllWindows()